In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from sklearn.decomposition import PCA
import xgboost as xgb
from bayes_opt import BayesianOptimization
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import time
from xgboost import XGBRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import log_loss
from sklearn import metrics
from pytorch_tabular.tabular_model import TabularModel
from pytorch_tabular.models.tab_transformer.config import TabTransformerConfig
from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig, ExperimentConfig
from sklearn.metrics import r2_score, mean_squared_error
from pytorch_tabular.models.tab_transformer.config import TabTransformerConfig
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import (RandomForestRegressor, AdaBoostRegressor,ExtraTreesRegressor)
from sklearn.preprocessing import MinMaxScaler, StandardScaler 
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Bidirectional 
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

dts = pd.read_csv("C:/Research/XTROP/Oral/oral_cancer_prediction_dataset.csv", sep=',')

# Filter for 'yes' in 'Oral Cancer (Diagnosis)'
dts['Oral Cancer (Diagnosis)'] = dts['Oral Cancer (Diagnosis)'].str.strip().str.lower()
dts = dts[dts['Oral Cancer (Diagnosis)'] == 'yes'].copy() # Use .copy() to avoid SettingWithCopyWarning

# Round off 'Survival Rate (5-Year, %)' to the nearest whole number (0 decimal places)
dts['Survival Rate (5-Year, %)'] = dts['Survival Rate (5-Year, %)'].round(0)

# Rename the column
dts.rename(columns={'Survival Rate (5-Year, %)': 'Survival Rate'}, inplace=True)

# Drop specified columns
columns_to_drop = ['ID', 'Cost of Treatment (USD)', 'Economic Burden (Lost Workdays per Year)', 'Early Diagnosis', 'Oral Cancer (Diagnosis)']
dts = dts.drop(columns=columns_to_drop)

# --- 2. Convert Categorical to Numeric using LabelEncoder ---
# Make a copy for numeric transformation
dts_numeric = dts.copy()

label_enc = LabelEncoder()

for col in dts_numeric.columns:
    if dts_numeric[col].dtype == 'object' or dts_numeric[col].dtype.name == 'category':
        dts_numeric[col] = label_enc.fit_transform(dts_numeric[col].astype(str))

# dts_numeric is now fully numeric
print("dts_numeric after Label Encoding (first 5 rows):")
print(dts_numeric.head())
print("\nData types after Label Encoding:")
print(dts_numeric.dtypes)


# --- 3. Split Data into Features (X) and Target (y) ---
y = dts_numeric['Survival Rate']
X = dts_numeric.drop(['Survival Rate'], axis=1) # inplace=False is default, can omit

print("\nX (features) head before scaling:")
print(X.head())

# --- 4. Split Data into Training and Testing Sets ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=0)

print("\nTraining set has {} samples.".format(X_train.shape[0]))
print("Testing set has {} samples.".format(X_test.shape[0]))

# --- 5. Apply MinMaxScaler (IMPORTANT: Fit on X_train, Transform on X_train and X_test) ---
scaler = MinMaxScaler()

# Fit the scaler only on the training data to prevent data leakage
X_train_scaled = scaler.fit_transform(X_train)
# Transform both training and testing data using the fitted scaler
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames with original column names
X_train = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

C:\Users\Lenovo\anaconda3\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


dts_numeric after Label Encoding (first 5 rows):
   Country  Age  Gender  Tobacco Use  Alcohol Consumption  HPV Infection  \
1        7   64       1            1                    1              1   
2       15   37       0            0                    1              0   
4       12   68       1            0                    0              0   
5       14   70       1            1                    0              1   
6       16   41       0            1                    1              0   

   Betel Quid Use  Chronic Sun Exposure  Poor Oral Hygiene  \
1               0                     1                  1   
2               0                     1                  1   
4               0                     0                  1   
5               1                     0                  1   
6               0                     0                  0   

   Diet (Fruits & Vegetables Intake)  Family History of Cancer  \
1                                  0                   

In [2]:
from sklearn.decomposition import PCA

# --- Re-running your previous PCA steps (ensure X_train and X_test are defined) ---
# Assuming X_train is a pandas DataFrame with feature names as columns
# For demonstration, let's create a dummy X_train if it's not defined
try:
    X_train.shape # Check if X_train exists
except NameError:
    print("X_train not found. Creating dummy data for demonstration.")
    # Create dummy data for demonstration if X_train is not available
    # In your actual notebook, ensure X_train is your preprocessed training data
    data = np.random.rand(100, 20)
    original_feature_names = [f'Feature_{i+1}' for i in range(20)]
    X_train = pd.DataFrame(data, columns=original_feature_names)
    X_test = pd.DataFrame(np.random.rand(50, 20), columns=original_feature_names)


print(f"Original feature count: {X_train.shape[1]}")

pca_full = PCA()
pca_full.fit(X_train)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
threshold = 0.95
n_components = np.argmax(cumulative_variance >= threshold) + 1
print(f"Number of PCA components to retain {threshold*100:.0f}% variance: {n_components}")

pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)
print(f"Reduced feature shape: {X_train_pca.shape[1]}")

# --- Get the original feature names ---
original_features = X_train.columns

# --- Display the details of each component (loadings) ---
print("\n--- Details of Each Principal Component (Loadings) ---")
for i, component in enumerate(pca.components_):
    print(f"\nPrincipal Component {i + 1}:")
    # Create a Series to map loadings to original feature names
    component_loadings = pd.Series(component, index=original_features)
    # Sort by absolute value to see the most influential features
    sorted_loadings = component_loadings.abs().sort_values(ascending=False)

    # Print top N contributing features for clarity (e.g., top 5)
    print(f"  Top 5 contributing features (by absolute loading):")
    for feature, abs_loading in sorted_loadings.items():
        # Get the actual loading value (not absolute)
        actual_loading = component_loadings[feature]
        print(f"    - {feature}: {actual_loading:.4f} (Absolute: {abs_loading:.4f})")

    # If you want to see all loadings for a component, uncomment the line below:
    # print(component_loadings.sort_values(ascending=False)) # Or sort by absolute value

# print("\nInterpretation: For each Principal Component, the 'loadings' indicate how much each original feature contributes to that component.
# A higher absolute value means a stronger contribution. 
# The sign indicates the direction of the relationship (e.g., a positive loading means that as the original feature's value increases, 
# the component's value also tends to increase, assuming other features are constant)")

Original feature count: 19
Number of PCA components to retain 95% variance: 17
Reduced feature shape: 17

--- Details of Each Principal Component (Loadings) ---

Principal Component 1:
  Top 5 contributing features (by absolute loading):
    - Poor Oral Hygiene: 0.9759 (Absolute: 0.9759)
    - Alcohol Consumption: 0.1981 (Absolute: 0.1981)
    - White or Red Patches in Mouth: -0.0660 (Absolute: 0.0660)
    - Gender: 0.0451 (Absolute: 0.0451)
    - HPV Infection: -0.0266 (Absolute: 0.0266)
    - Difficulty Swallowing: 0.0230 (Absolute: 0.0230)
    - Betel Quid Use: -0.0183 (Absolute: 0.0183)
    - Oral Lesions: 0.0162 (Absolute: 0.0162)
    - Chronic Sun Exposure: 0.0050 (Absolute: 0.0050)
    - Diet (Fruits & Vegetables Intake): -0.0046 (Absolute: 0.0046)
    - Compromised Immune System: 0.0029 (Absolute: 0.0029)
    - Treatment Type: 0.0029 (Absolute: 0.0029)
    - Tumor Size (cm): 0.0029 (Absolute: 0.0029)
    - Unexplained Bleeding: 0.0026 (Absolute: 0.0026)
    - Age: 0.0019 (Absol

In [12]:
from bayes_opt import BayesianOptimization
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
import time

# Dummy data section (replace with your real X, y, and PCA if needed)
# X_train, X_test, y_train, y_test should already be defined

# Define optimization function
def optimize_decision_tree(max_depth, min_samples_split):
    model = DecisionTreeRegressor(max_depth=int(max_depth), min_samples_split=int(min_samples_split))
    model.fit(X_train, y_train)
    return r2_score(y_test, model.predict(X_test))

# Run Bayesian Optimization
bo = BayesianOptimization(
    f=optimize_decision_tree,
    pbounds={'max_depth': (2, 20), 'min_samples_split': (2, 20)},
    random_state=42
)
bo.maximize(init_points=3, n_iter=5)

# Extract best params
best = bo.max['params']
best_params = {
    "Decision Tree": {
        "max_depth": best['max_depth'],
        "min_samples_split": best['min_samples_split']
    }
}

# Train final model
model = DecisionTreeRegressor(
    max_depth=int(best['max_depth']),
    min_samples_split=int(best['min_samples_split'])
)

start = time.time()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
end = time.time()

# Evaluation metrics
result = pd.DataFrame([{
    "Model": "DecisionTreeRegressor",
    "R² Score": r2_score(y_test, y_pred),
    "MAE": mean_absolute_error(y_test, y_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "Run Time (s)": round(end - start, 4)
}])
print(result)

# ✅ Format best hyperparameters
best_hyperparams = []

for model_name, params in best_params.items():
    param_record = {"Model": model_name}
    for param_key, value in params.items():
        param_record[param_key] = round(value, 6) if isinstance(value, float) else int(value)
    best_hyperparams.append(param_record)

best_hyperparams_df = pd.DataFrame(best_hyperparams)

# ✅ Display the table
print("\n🔧 Best Hyperparameters for Each Model:")
print(best_hyperparams_df.set_index("Model"))


|   iter    |  target   | max_depth | min_sa... |
-------------------------------------------------
| 1         | 0.9609    | 8.742     | 19.11     |
| 2         | 0.9559    | 15.18     | 12.78     |
| 3         | 0.9613    | 4.808     | 4.808     |
| 4         | 0.9609    | 8.731     | 19.0      |
| 5         | 0.9613    | 2.0       | 12.9      |
| 6         | 0.9613    | 2.0       | 20.0      |
| 7         | 0.9613    | 2.0       | 2.0       |
| 8         | 0.9423    | 20.0      | 2.0       |
                   Model  R² Score       MAE      RMSE  Run Time (s)
0  DecisionTreeRegressor  0.961339  3.869018  4.651269        0.0729

🔧 Best Hyperparameters for Each Model:
               max_depth  min_samples_split
Model                                      
Decision Tree        2.0          12.899565


In [13]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import time

# ✔️ Use your existing PCA-transformed train/test data
# Assuming: X_train_pca, X_test_pca, y_train, y_test are already defined

# ✅ Round `min_samples_split` to at least 2
model = DecisionTreeRegressor(
    max_depth=2,
    min_samples_split=max(int(round(12.899565)), 2)
)

# ⏱️ Train and evaluate
start = time.time()
model.fit(X_train_pca, y_train)
y_pred = model.predict(X_test_pca)
end = time.time()

# 📊 Evaluation metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
runtime = round(end - start, 4)

# 📋 Print results
print(f"R² Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Run Time: {runtime} seconds")


R² Score: 0.9613
MAE: 3.8690
RMSE: 4.6513
Run Time: 0.4169 seconds


In [16]:
import eli5
from eli5.sklearn import PermutationImportance
from IPython.display import display

# ✅ Step 1: Fit Permutation Importance (Already trained DecisionTreeRegressor on PCA data)
perm = PermutationImportance(model, random_state=42, scoring='r2')
perm.fit(X_test_pca, y_test)

# ✅ Step 2: Display feature importance using show_weights()
display(eli5.show_weights(perm, feature_names=[f"PC{i+1}" for i in range(X_test_pca.shape[1])]))

# ✅ Step 3: Use explain_weights() separately if you want to extract the explanation object
explanation = eli5.explain_weights(perm, feature_names=[f"PC{i+1}" for i in range(X_test_pca.shape[1])])

# ✅ Optional: Display explanation as DataFrame
from eli5.formatters import format_as_dataframe
weights_df = format_as_dataframe(explanation)
print(weights_df)


Weight,Feature
1.9189 ± 0.0685,PC14
0 ± 0.0000,PC17
0 ± 0.0000,PC8
0 ± 0.0000,PC2
0 ± 0.0000,PC3
0 ± 0.0000,PC4
0 ± 0.0000,PC5
0 ± 0.0000,PC6
0 ± 0.0000,PC7
0 ± 0.0000,PC9


   feature    weight       std
0     PC14  1.918868  0.034236
1     PC17  0.000000  0.000000
2      PC8  0.000000  0.000000
3      PC2  0.000000  0.000000
4      PC3  0.000000  0.000000
5      PC4  0.000000  0.000000
6      PC5  0.000000  0.000000
7      PC6  0.000000  0.000000
8      PC7  0.000000  0.000000
9      PC9  0.000000  0.000000
10    PC16  0.000000  0.000000
11    PC10  0.000000  0.000000
12    PC11  0.000000  0.000000
13    PC12  0.000000  0.000000
14    PC13  0.000000  0.000000
15    PC15  0.000000  0.000000
16     PC1  0.000000  0.000000
